# 14 · Summarize points to polygons, checked against shapely

## What this checks

**Summarize points to polygons** decides which locations fall in which polygon. Then it counts them and totals a value
for each polygon. This check covers release 0.5.0.

In the John Snow example, this answers the real question: how many deaths fall nearest each pump. A location placed
in the wrong polygon moves a death from one pump to another. So it is worth checking.

## How it is checked

Fieldwork decides membership with its own test of whether a point is inside a polygon. We decide it again with
**shapely**, the Python library for shapes. Shapely uses GEOS, the geometry library behind most GIS software. No code
is shared with Fieldwork.

The hard part is a location on a polygon's edge. Fieldwork's boundary setting decides it:

- **`include`** counts a location inside the polygon or on its edge. In GEOS, that test is called "covers".
- **`exclude`** counts only a location strictly inside. In GEOS, that test is called "contains".

Mixing the two up would give a check that passes on data with no edge cases, and fails as soon as one appears. So the
pairing is stated here. The number of locations on an edge is reported, whether or not there are any.

The data are 250 locations of deaths in 8 polygons, one around each pump, from the John Snow example. The setting is
`include`.

## What should happen

1. **The count in each polygon** matches exactly. Counts are whole numbers, so there is no tolerance.
2. **The total in each polygon**, the sum of the declared value, matches exactly.
3. **The missing values in each polygon** match exactly. A location with no value, or a value that is not a number,
   adds nothing, and must be counted.
4. **The summary** matches exactly. It counts records, matched and unmatched locations, locations with no
   coordinates, and locations in more than one polygon.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports.
NEEDED = ["shapely"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_14_summarize_polygons import run

report = run(Path.cwd() / "fixtures" / "summarize-polygons-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["points"], "locations in", report["polygons"], "polygons")
print()
print(f"{'polygon':<12} {'count':>7} {'shapely':>8} {'total':>8} {'shapely':>8}")
for row in report["perPolygon"]:
    print(f"{row['id']:<12} {row['fieldworkCount']:>7} {row['referenceCount']:>8} "
          f"{row['fieldworkTotal']:>8} {row['referenceTotal']:>8}")
print()
print("summary, fieldwork:", {k: v for k, v in report["fieldwork"].items() if k != "valueField"})
print("summary, shapely:  ", {k: v for k, v in report["reference"].items() if k != "valueField"})

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.5.0:
**Agrees**.

Every count and every total in all 8 polygons matched exactly, over 250 locations. The summary matched exactly. The
largest difference in any total was 0.

On 28 October 2026 the data were made again from release 0.2.0, and the inputs and outputs were identical to the
first run's.

Running the cell above repeats the check on the same data.

## What this data does not test

**No location in this data lies on a polygon edge**, and none is in more than one polygon. So the difference between
"covers" and "contains" is stated, but not tested, by the check above. That is the part of this widget most likely to
go wrong.

It is better to say so than to let a passing result suggest otherwise. The cell below builds the case the data lack.

In [ ]:
from shapely.geometry import Point, Polygon

# Two cells meeting along x = 0, and a location exactly on the shared edge.
left = Polygon([(-1, -1), (0, -1), (0, 1), (-1, 1)])
right = Polygon([(0, -1), (1, -1), (1, 1), (0, 1)])
on_edge = Point(0, 0)
inside_left = Point(-0.5, 0)

for label, geos, predicate in [("include", "covers", lambda p, q: p.covers(q)),
                               ("exclude", "contains", lambda p, q: p.contains(q))]:
    hits = [name for name, cell in (("left", left), ("right", right)) if predicate(cell, on_edge)]
    print(f"boundary={label:<8} (GEOS {geos:<8}): the edge location belongs to {hits or ['nothing']}")

print()
print("So under 'include' a single location is counted twice -- once for each cell -- and under")
print("'exclude' it is counted not at all and reported as unmatched. Both are defensible and")
print("they are different numbers, which is why the policy is a declared parameter and travels")
print("in the fixture rather than being assumed.")
print()
print(f"A control: a location well inside the left cell belongs to "
      f"{[n for n, c in (('left', left), ('right', right)) if c.covers(inside_left)]} either way.")

## What this does not show

- That nearest-pump polygons show which pump anybody drank from. They assume each person used the nearest pump in a
  straight line. That is what the widget computes, not what anybody did.
- That the per-polygon totals can be added. Every membership is kept, so an overlapping location counts more than
  once.
- That locations on an edge are handled correctly. This data has none.